In [1]:
import numpy as np

def convertible_bond(S0, r, sigma, T, N, face=100, credit_spread=0.02):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u

    r_adj = r + credit_spread
    p = (np.exp(r * dt) - d) / (u - d)

    stock = np.zeros((N+1, N+1))
    bond = np.zeros((N+1, N+1))

    # Stock tree
    for i in range(N+1):
        for j in range(i+1):
            stock[j, i] = S0 * (u**(i-j)) * (d**j)

    # Payoff at maturity
    for j in range(N+1):
        bond[j, N] = max(face, stock[j, N])

    # Backward induction
    for i in range(N-1, -1, -1):
        for j in range(i+1):
            hold = np.exp(-r_adj*dt) * (p*bond[j, i+1] + (1-p)*bond[j+1, i+1])
            convert = stock[j, i]
            bond[j, i] = max(hold, convert)

    # Delta (Greek)
    delta = (bond[0,1] - bond[1,1]) / (stock[0,1] - stock[1,1])

    return bond[0,0], delta, stock, bond